<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">


# Python for Finance, 3rd Edition
## Chapter 25 · Automated Deployment of Trading Strategies

&copy; Dr. Yves J. Hilpisch<br>
AI-supported by GPT 5.x<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh


## Notebook Goals
This notebook mirrors the Chapter 25 deployment workflow in an interactive
format. It rebuilds the baseline ML logic locally, runs the historical
`engine` bridge, and then inspects the simulated live deployment step by step.


### How to Use This Notebook
- Run the cells from top to bottom the first time.
- The setup cell switches into the project root and adds `code/` to
  `sys.path`.
- The notebook defines the deployment helpers itself instead of importing
  the chapter module.


### Notebook Setup
Move to the project root first so that the notebook can reuse the same
relative paths and local packages as the chapter scripts.


In [ ]:
from pathlib import Path  # filesystem paths
import os  # working-directory handling
import sys  # local package imports

PROJECT_ROOT = Path.cwd().resolve()  # current notebook location
if not (PROJECT_ROOT / "data" / "eod_data.csv").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent  # support launches from notebooks/
os.chdir(PROJECT_ROOT)  # switch to the book project root

CODE_PATH = PROJECT_ROOT / "code"
if str(CODE_PATH) not in sys.path:
    sys.path.insert(0, str(CODE_PATH))

Path.cwd()


## Imports and Engine API
Import the modelling tools and the public `engine` API used for both the
historical replay bridge and the simulated live deployment.


In [ ]:
import json
from dataclasses import dataclass
from typing import Any

import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from engine import HistoricalFeed, PaperBroker, TradingSession, build_feed

TMP_DIR = PROJECT_ROOT / "_tmp" / "ch25"


## Historical Prices Through the Engine
The deployment example should not bypass the `engine`. Start from a historical
feed and access the exposed close series through `feed.prices`.


In [ ]:
feed = HistoricalFeed.from_symbol("EURUSD", spread_bps=1.0)
closes = feed.prices.iloc[-130:]
closes.tail()


## Baseline ML Helpers
Rebuild the baseline logistic-regression pipeline, feature engineering, and
latest-row construction directly in the notebook.


In [ ]:
def build_logistic_regression_pipeline() -> Pipeline:
    """Create the baseline logistic-regression pipeline."""

    return Pipeline(
        steps=[
            ("scaler", StandardScaler()),
            (
                "logreg",
                LogisticRegression(
                    C=10.0,
                    solver="lbfgs",
                    max_iter=500,
                    random_state=0,
                ),
            ),
        ],
    )


def make_features_and_labels(
    closes: pd.Series,
) -> tuple[pd.DataFrame, pd.Series]:
    """Construct lagged features and next-bar direction labels."""

    rets = closes.pct_change()
    next_ret = closes.shift(-1) / closes - 1.0
    data = pd.DataFrame(
        {
            "r_lag1": rets.shift(1),
            "mom_3": rets.rolling(3).mean(),
            "mom_5": rets.rolling(5).mean(),
            "next_ret": next_ret,
        }
    ).dropna()

    X = data[["r_lag1", "mom_3", "mom_5"]]
    y = (data["next_ret"] > 0.0).astype(int)
    return X, y


def make_latest_feature_row(closes: pd.Series) -> pd.DataFrame | None:
    """Construct the latest feature row for the next prediction."""

    rets = closes.pct_change()
    row = pd.DataFrame(
        {
            "r_lag1": [rets.shift(1).iloc[-1]],
            "mom_3": [rets.rolling(3).mean().iloc[-1]],
            "mom_5": [rets.rolling(5).mean().iloc[-1]],
        },
        index=[closes.index[-1]],
    )
    if row.isna().any(axis=None):
        return None
    return row


In [ ]:
X, y = make_features_and_labels(closes)
X.tail()


In [ ]:
y.tail()


## Deployment Configuration and Logger
Deployment needs explicit timing, sizing, and logging assumptions.


In [ ]:
# Configuration dataclass and structured logger used by the deployer.

@dataclass
class DeploymentConfig:
    sample_interval: str
    training_window: int
    retrain_interval: int
    position_size: float
    probability_threshold: float = 0.5


class DeploymentLogger:
    """Collect structured deployment events and optionally persist them."""

    def __init__(self, name: str) -> None:
        self.name = name
        self.events: list[dict[str, Any]] = []

    def log(
        self,
        event: str,
        timestamp: pd.Timestamp,
        **payload: Any,
    ) -> None:
        self.events.append(
            {
                "logger": self.name,
                "event": event,
                "timestamp": pd.Timestamp(timestamp),
                **payload,
            }
        )

    def to_frame(self) -> pd.DataFrame:
        frame = pd.DataFrame(self.events)
        if frame.empty:
            return frame
        return frame.sort_values("timestamp").reset_index(drop=True)

    def write_jsonl(self, path: Path) -> Path:
        path.parent.mkdir(parents=True, exist_ok=True)
        with path.open("w", encoding="utf-8") as handle:
            for event in self.events:
                payload = dict(event)
                payload["timestamp"] = pd.Timestamp(
                    payload["timestamp"]
                ).isoformat()
                handle.write(json.dumps(payload) + "\n")
        return path


## Deployable Strategy Class
Define the stateful strategy object that collects ticks, resamples them to
bars, refits the model, and rebalances only when the target position changes.


In [ ]:
class BaselineMLDeployer:
    """Deploy the chapter 23 baseline model on resampled bars."""

    def __init__(
        self,
        config: DeploymentConfig,
        logger: DeploymentLogger,
    ) -> None:
        self.config = config
        self.logger = logger
        self.tick_history: list[dict[str, object]] = []
        self.latest_bar_time: pd.Timestamp | None = None
        self.latest_signal: int = 0
        self.latest_probability: float | None = None
        self.model: Pipeline | None = None
        self.last_fit_bar_count = 0

    def on_tick(self, session: TradingSession, tick) -> None:
        self.tick_history.append(
            {
                "timestamp": pd.Timestamp(tick.timestamp),
                "mid": tick.mid,
            }
        )
        bars = self._completed_bars()
        if bars.empty:
            return

        bar_time = pd.Timestamp(bars.index[-1])
        is_stale = (
            self.latest_bar_time is not None
            and bar_time <= self.latest_bar_time
        )
        if is_stale:
            return

        self.latest_bar_time = bar_time
        self.logger.log(
            "bar",
            bar_time,
            close=float(bars.iloc[-1]),
            bars_seen=int(len(bars)),
        )

        X_trainable, y_trainable = make_features_and_labels(bars)
        if len(X_trainable) < self.config.training_window:
            self.logger.log(
                "warmup",
                bar_time,
                bars_seen=int(len(bars)),
                rows_available=int(len(X_trainable)),
                rows_required=self.config.training_window,
            )
            return

        if (
            self.model is None
            or len(bars) - self.last_fit_bar_count
            >= self.config.retrain_interval
        ):
            self._fit_model(X_trainable, y_trainable, bars_seen=len(bars))
            self.logger.log(
                "model_refit",
                bar_time,
                bars_used=self.last_fit_bar_count,
            )

        feature_row = make_latest_feature_row(bars)
        if feature_row is None:
            return

        proba_up = float(self.model.predict_proba(feature_row.values)[0, 1])
        signal = 1 if proba_up >= self.config.probability_threshold else -1
        self.latest_probability = proba_up
        self.latest_signal = signal
        self.logger.log(
            "signal",
            bar_time,
            proba_up=round(proba_up, 6),
            signal=signal,
        )

        self._rebalance(session=session, timestamp=bar_time)

    def _completed_bars(self) -> pd.Series:
        ticks = (
            pd.DataFrame(self.tick_history)
            .set_index("timestamp")
            .sort_index()
        )
        bars = (
            ticks["mid"]
            .resample(self.config.sample_interval)
            .last()
            .dropna()
        )
        if len(bars) <= 1:
            return pd.Series(dtype=float)
        return bars.iloc[:-1]

    def completed_bars(self) -> pd.Series:
        return self._completed_bars().copy()

    def _fit_model(
        self,
        X_trainable: pd.DataFrame,
        y_trainable: pd.Series,
        bars_seen: int,
    ) -> None:
        X_train = X_trainable.iloc[-self.config.training_window:]
        y_train = y_trainable.iloc[-self.config.training_window:]
        self.model = build_logistic_regression_pipeline()
        self.model.fit(X_train.values, y_train.values)
        self.last_fit_bar_count = int(bars_seen)

    def _rebalance(
        self,
        session: TradingSession,
        timestamp: pd.Timestamp,
    ) -> None:
        positions = session.broker.get_positions()
        current_quantity = (
            0.0 if not positions else float(positions[0]["quantity"])
        )
        target_quantity = self.config.position_size * float(self.latest_signal)
        delta = target_quantity - current_quantity
        if abs(delta) < 1e-12:
            self.logger.log(
                "hold",
                timestamp,
                target_quantity=target_quantity,
            )
            return

        side = "buy" if delta > 0.0 else "sell"
        receipt = session.place_market_order(
            symbol="EURUSD",
            side=side,
            quantity=abs(delta),
            meta={
                "signal": self.latest_signal,
                "proba_up": round(float(self.latest_probability), 6),
                "bar_timestamp": timestamp.isoformat(),
            },
        )
        snapshot = receipt["account_snapshot"]
        self.logger.log(
            "order_submitted",
            timestamp,
            side=side,
            quantity=abs(delta),
            fill_price=round(float(receipt["fill"]["price"]), 6),
            equity=round(float(snapshot["equity"]), 6),
        )


def build_monitoring_report(
    session: TradingSession,
    logger: DeploymentLogger,
) -> pd.Series:
    logs = logger.to_frame()
    snapshot = session.broker.get_account_snapshot()
    signal_logs = logs.loc[logs["event"] == "signal", "signal"]

    return pd.Series(
        {
            "bars_processed": int((logs["event"] == "bar").sum()),
            "model_refits": int((logs["event"] == "model_refit").sum()),
            "orders_submitted": int((logs["event"] == "order_submitted").sum()),
            "last_signal": (
                int(signal_logs.iloc[-1])
                if not signal_logs.empty
                else 0
            ),
            "final_equity": float(snapshot["equity"]),
            "realized_pnl": float(snapshot["realized_pnl"]),
            "open_positions": int(len(snapshot["positions"])),
        }
    )


## Historical Replay Through the Engine
The first run is the bridge from event-based backtesting to deployment.


In [ ]:
# Convenience wrappers that run the two end-to-end deployment examples.

def run_historical_engine_bridge() -> dict[str, object]:
    base_feed = HistoricalFeed.from_symbol("EURUSD", spread_bps=1.0)
    prices = base_feed.prices.iloc[-260:]
    feed = HistoricalFeed(
        symbol=base_feed.symbol,
        prices=prices,
        spread_bps=base_feed.spread_bps,
    )
    broker = PaperBroker(initial_cash=50_000.0, account_id="HIST-001")
    session = TradingSession(feed=feed, broker=broker)
    logger = DeploymentLogger(name="historical_bridge")
    deployer = BaselineMLDeployer(
        config=DeploymentConfig(
            sample_interval="1B",
            training_window=120,
            retrain_interval=20,
            position_size=10.0,
        ),
        logger=logger,
    )
    history = session.run(deployer.on_tick)
    return {
        "session": session,
        "history": history,
        "logs": logger.to_frame(),
        "report": build_monitoring_report(session=session, logger=logger),
        "bars": deployer.completed_bars(),
        "receipts": list(session.receipts),
        "logger": logger,
    }


def run_simulated_live_deployment() -> dict[str, object]:
    feed = build_feed(
        symbol="EURUSD",
        mode="simulated",
        periods=360,
        seed=11,
        start="2027-01-04 09:00:00",
        base_interval="1s",
        arrival_model="jittered",
        jitter_seconds=0.35,
        spread_bps=1.0,
    )
    broker = PaperBroker(initial_cash=50_000.0, account_id="SIM-001")
    session = TradingSession(feed=feed, broker=broker)
    logger = DeploymentLogger(name="simulated_live")
    deployer = BaselineMLDeployer(
        config=DeploymentConfig(
            sample_interval="5s",
            training_window=40,
            retrain_interval=10,
            position_size=5.0,
        ),
        logger=logger,
    )
    history = session.run(deployer.on_tick)
    return {
        "session": session,
        "history": history,
        "logs": logger.to_frame(),
        "report": build_monitoring_report(session=session, logger=logger),
        "bars": deployer.completed_bars(),
        "receipts": list(session.receipts),
        "logger": logger,
    }


In [ ]:
historical = run_historical_engine_bridge()
historical["report"].round(6)


In [ ]:
historical["bars"].tail()


In [ ]:
cols = [
    "timestamp",
    "event",
    "bars_seen",
    "rows_available",
    "rows_required",
    "bars_used",
    "proba_up",
    "signal",
    "side",
    "quantity",
    "equity",
]
mask = historical["logs"]["event"].isin(
    ["warmup", "model_refit", "signal", "order_submitted"]
)
historical["logs"].loc[mask].reindex(columns=cols).head(12)


In [ ]:
historical["receipts"][0]


## Simulated Live Deployment
Switch only the feed configuration so the same strategy now consumes irregular
simulated ticks and resamples them to five-second bars.


In [ ]:
simulated = run_simulated_live_deployment()
simulated["report"].round(6)


In [ ]:
simulated["history"][
    ["bid", "ask", "mid", "cash", "equity", "position_quantity"]
].head()


In [ ]:
simulated["bars"].head(10)


In [ ]:
simulated["logs"].loc[mask].reindex(columns=cols).tail(12)


In [ ]:
simulated["receipts"][-1]


## Persisted Structured Logs
The deployment logger can also write events to `JSONL`, which is useful for
monitoring, auditing, and replaying an automated run later.


In [ ]:
logger = DeploymentLogger(name="notebook_demo")
logger.log("heartbeat", pd.Timestamp("2027-01-04 09:00:00"), status="ok")
path = logger.write_jsonl(TMP_DIR / "notebook_demo.jsonl")
path
